# LAB 08 — Support Vector Machines
## Student Information

- **Name:** Muhammad Umar
- **Student ID:** 023-24-0260
- **Section:** H
- **GitHub:** Not provided
- **Kaggle:** Not provided
- **Dataset:** Telco Customer Churn (Kaggle)
- **Dataset Source:** https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Environment:** This notebook is written for Jupyter Notebook / VS Code. It uses the dataset path supplied by the student. If that path is unavailable, the uploaded dataset path is used as a fallback in this working environment.

**Experiment settings:** `test_size=0.20`, `random_state=42`, `stratify=y`.

## Part 0 — Recap and Setup

The dataset is loaded, `customerID` is removed, `TotalCharges` is converted to numeric, missing values are handled using the training-set median, and `Churn` is mapped to 0/1.

To avoid data leakage, the raw data is split first. One-hot encoding is then prepared from the training features and the test columns are aligned to the training columns. The scaler is also fitted only on training data.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score,
    ConfusionMatrixDisplay
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
TEST_SIZE = 0.20

dataset_path = r"C:\Users\umarc\Desktop\ML-Lab Practice\datasets\Telco-Customer-Churn.csv"
if not os.path.exists(dataset_path):
    dataset_path = "/mnt/data/Telco-Customer-Churn(1).csv"

df = pd.read_csv(dataset_path)
print("Dataset shape:", df.shape)
display(df.head())
print("\nChurn distribution:")
display(df["Churn"].value_counts())

In [ ]:
data = df.copy()
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
data = data.drop(columns=["customerID"])
data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

X = data.drop(columns=["Churn"])
y = data["Churn"]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

total_charges_median = X_train_raw["TotalCharges"].median()
X_train_raw = X_train_raw.copy()
X_test_raw = X_test_raw.copy()
X_train_raw["TotalCharges"] = X_train_raw["TotalCharges"].fillna(total_charges_median)
X_test_raw["TotalCharges"] = X_test_raw["TotalCharges"].fillna(total_charges_median)

X_train = pd.get_dummies(X_train_raw, drop_first=False, dtype=int)
X_test = pd.get_dummies(X_test_raw, drop_first=False, dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Missing TotalCharges before imputation:", df["TotalCharges"].astype(str).str.strip().eq("").sum())
print("Missing values after preprocessing:", int(X_train.isna().sum().sum() + X_test.isna().sum().sum()))
print("Churn rate in training set:", round(y_train.mean(), 4))
print("Churn rate in test set:", round(y_test.mean(), 4))

**Part 0 interpretation:** The dataset contains customer information and a binary churn target. The train/test split is stratified so the churn proportion remains approximately consistent in both sets. `TotalCharges` is imputed with the training median, and the scaler is fitted only on training data.

## Part 1 — Why Scaling Matters for SVM

Two linear SVM models are trained on the same split: one on unscaled features and one on standardized features. Accuracy, F1-score, training time, and convergence warnings are compared.

In [ ]:
part1_rows = []
part1_warnings = {}

for label, train_data, test_data in [
    ("Unscaled", X_train, X_test),
    ("Scaled", X_train_s, X_test_s)
]:
    model = SVC(kernel="linear", random_state=RANDOM_STATE, max_iter=20000)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        start = time.time()
        model.fit(train_data, y_train)
        elapsed = time.time() - start
    pred = model.predict(test_data)
    part1_rows.append({
        "Model": label,
        "Accuracy": accuracy_score(y_test, pred),
        "F1-score": f1_score(y_test, pred),
        "Training Time (seconds)": elapsed,
        "Warnings": len(caught)
    })
    part1_warnings[label] = [str(w.message) for w in caught]

part1_results = pd.DataFrame(part1_rows)
display(part1_results.round(4))
print("Warnings:", part1_warnings)

In [ ]:
raw_row = part1_results.set_index("Model").loc["Unscaled"]
scaled_row = part1_results.set_index("Model").loc["Scaled"]
print(
    f"(a) Unscaled accuracy/F1 = {raw_row['Accuracy']:.4f}/{raw_row['F1-score']:.4f}; "
    f"scaled accuracy/F1 = {scaled_row['Accuracy']:.4f}/{scaled_row['F1-score']:.4f}. "
    f"Training times were {raw_row['Training Time (seconds)']:.2f}s and "
    f"{scaled_row['Training Time (seconds)']:.2f}s respectively. "
    f"Warnings: unscaled={int(raw_row['Warnings'])}, scaled={int(scaled_row['Warnings'])}."
)
print("(b) SVM uses distances and geometric margins. Large-range features can dominate the margin calculation, "
      "so the decision boundary may be biased when features are not scaled.")
print("(c) Standardized features will be used for the remaining SVM experiments because they put numeric "
      "features on comparable scales and make the margin calculation more balanced.")

## Part 2 — Kernel Comparison

Linear, RBF, and polynomial SVMs are trained on the same standardized training data. The comparison includes accuracy, precision, recall, F1-score, and total support-vector count.

In [ ]:
kernels = ["linear", "rbf", "poly"]
kernel_models = {}
kernel_rows = []

for kernel in kernels:
    model = SVC(kernel=kernel, random_state=RANDOM_STATE)
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    kernel_models[kernel] = model
    kernel_rows.append({
        "Kernel": kernel,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "Support Vectors": model.support_.shape[0]
    })

kernel_results = pd.DataFrame(kernel_rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
display(kernel_results.round(4))
best_kernel_name = kernel_results.loc[0, "Kernel"]
best_kernel_model = kernel_models[best_kernel_name]
best_kernel_pred = best_kernel_model.predict(X_test_s)

ConfusionMatrixDisplay.from_predictions(y_test, best_kernel_pred, cmap="Blues")
plt.title(f"Confusion Matrix — Best Kernel ({best_kernel_name})")
plt.show()
print(f"Best kernel by F1-score: {best_kernel_name}")

In [ ]:
most_sv_row = kernel_results.loc[kernel_results["Support Vectors"].idxmax()]
best_kernel_row = kernel_results.iloc[0]
print(f"(a) {best_kernel_name} produced the highest test F1-score "
      f"({best_kernel_row['F1-score']:.4f}). This is an empirical result; a more flexible kernel is not "
      "automatically guaranteed to perform best.")
print(f"(b) {most_sv_row['Kernel']} used the most support vectors "
      f"({int(most_sv_row['Support Vectors'])}). A larger count means more training points define or influence "
      "the learned boundary; it does not by itself prove overfitting.")
print(f"(c) The {best_kernel_name} kernel is the strongest candidate from this comparison by test F1-score. "
      "The RBF kernel is still tuned in Part 4 as specifically required by the lab manual.")

**Confusion-matrix interpretation:** The matrix above shows correct and incorrect churn predictions for the best kernel by test F1-score. The lower-left cell represents churners missed by the model (false negatives), while the upper-right cell represents loyal customers incorrectly flagged as churners (false positives).

## Part 3 — Visualizing the Margin: What Does C Actually Do?

PCA reduces the standardized training features to two components. Three RBF SVMs are trained on this 2-D projection using small, medium, and large C values. These plots are an illustration only; they are not the full high-dimensional model.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_s)
explained_variance = pca.explained_variance_ratio_.sum()

print("Explained variance ratio (two components):", round(explained_variance, 4))

C_values = [0.01, 1, 100]
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 250),
    np.linspace(y_min, y_max, 250)
)

part3_support = {}
part3_train_accuracy = {}
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, c_value in zip(axes, C_values):
    model = SVC(kernel="rbf", C=c_value, gamma="scale")
    model.fit(X_train_pca, y_train)
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid_points).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
    ax.scatter(X_train_pca[:, 0], X_train_pca[:, 1], c=y_train, cmap="coolwarm",
               edgecolors="k", s=12, alpha=0.65)
    ax.set_title(f"C = {c_value} | Support vectors = {len(model.support_)}")
    ax.set_xlabel("Principal Component 1")
    ax.set_ylabel("Principal Component 2")
    part3_support[c_value] = len(model.support_)
    part3_train_accuracy[c_value] = accuracy_score(y_train, model.predict(X_train_pca))

plt.tight_layout()
plt.show()
print("Two-component explained variance:", round(explained_variance, 4))
print("Support vectors by C:", part3_support)
print("Training accuracy by C:", {c: round(v, 4) for c, v in part3_train_accuracy.items()})

In [ ]:
lowest_c = min(C_values, key=lambda c: part3_train_accuracy[c])
highest_c = max(C_values, key=lambda c: part3_train_accuracy[c])
print("(a) Increasing C makes margin violations more costly. Compare the three plotted boundaries: a larger C can create a more detailed boundary to fit training points, while a smaller C allows a wider margin and more violations.")
print(f"(b) Support-vector counts for C={C_values} are {[part3_support[c] for c in C_values]}. The counts are data-dependent and need not decrease monotonically as C changes.")
print(f"(c) The lowest training accuracy occurred at C={lowest_c} ({part3_train_accuracy[lowest_c]:.4f}); the highest occurred at C={highest_c} ({part3_train_accuracy[highest_c]:.4f}). A low-C model may underfit if its boundary is too simple; a high-C model may overfit if its detailed boundary fits noise. The 2-D projection is only an illustration, so use the plots and full-feature CV results together.")
print(f"(d) PCA's two components explain {explained_variance:.2%} of the training-feature variance. The omitted dimensions may contain useful class information, so this 2-D boundary can differ from the real full-feature SVM boundary.")

## Part 4 — Systematic Hyperparameter Search

GridSearchCV tests multiple C and gamma combinations with 5-fold cross-validation. The selection metric is F1-score, as required by the manual. The final model is evaluated once on the held-out test set.

In [ ]:
param_grid = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", 0.01, 0.1]
}

grid_search = GridSearchCV(
    SVC(kernel="rbf"),
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    return_train_score=True
)

grid_start = time.time()
grid_search.fit(X_train_s, y_train)
grid_time = time.time() - grid_start

best_svm = grid_search.best_estimator_
best_svm_pred = best_svm.predict(X_test_s)

grid_results = pd.DataFrame(grid_search.cv_results_)
heatmap_data = grid_results.pivot(index="param_C", columns="param_gamma", values="mean_test_score")
heatmap_data.index = heatmap_data.index.astype(str)

plt.figure(figsize=(8, 5))
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("GridSearchCV Mean F1-score — RBF SVM")
plt.xlabel("Gamma")
plt.ylabel("C")
plt.show()

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation F1-score:", round(grid_search.best_score_, 4))
print("Grid search time (seconds):", round(grid_time, 2))
print("Test accuracy:", round(accuracy_score(y_test, best_svm_pred), 4))
print("Test precision:", round(precision_score(y_test, best_svm_pred, zero_division=0), 4))
print("Test recall:", round(recall_score(y_test, best_svm_pred, zero_division=0), 4))
print("Test F1-score:", round(f1_score(y_test, best_svm_pred, zero_division=0), 4))
print("Test confusion matrix:\n", confusion_matrix(y_test, best_svm_pred))

In [ ]:
cv_test_gap = abs(grid_search.best_score_ - f1_score(y_test, best_svm_pred))
untuned_rbf_f1 = kernel_results.loc[kernel_results["Kernel"] == "rbf", "F1-score"].iloc[0]
tuned_rbf_f1 = f1_score(y_test, best_svm_pred)

print(f"(a) Best C={grid_search.best_params_['C']}, gamma={grid_search.best_params_['gamma']}, "
      f"cross-validated F1={grid_search.best_score_:.4f}.")
best_cv = grid_search.best_score_
near_best = grid_results[grid_results["mean_test_score"] >= best_cv - 0.01][["param_C", "param_gamma", "mean_test_score"]]
print("(b) Combinations within 0.01 of the best CV F1-score:")
display(near_best.sort_values("mean_test_score", ascending=False))
print("If several neighboring combinations appear above, performance is relatively stable in that region; if only one appears, the model is more sensitive to the exact settings.")
print(f"(c) CV F1={grid_search.best_score_:.4f}; test F1={tuned_rbf_f1:.4f}; absolute gap={cv_test_gap:.4f}. "
      "A large gap can indicate overfitting to the cross-validation folds or a test set that differs from them.")
print(f"(d) Untuned RBF test F1={untuned_rbf_f1:.4f}; tuned RBF test F1={tuned_rbf_f1:.4f}. "
      f"Change={tuned_rbf_f1-untuned_rbf_f1:+.4f}. Tuning is worthwhile if the gain justifies its extra computation.")

**Heatmap interpretation:** Each cell is the mean validation F1-score for one C/gamma pair. The highest cell identifies the selected setting. The held-out test set is kept separate from this search and is used only for final evaluation.

## Part 5 — Class Imbalance and the Business Trade-off

The best C and gamma from Part 4 are used to train two RBF SVMs: the default model and a model with `class_weight='balanced'`. `probability=True` is enabled so probability-like churn scores can be used for the ROC curve.

In [ ]:
best_C = grid_search.best_params_["C"]
best_gamma = grid_search.best_params_["gamma"]

svm_default_prob = SVC(
    kernel="rbf", C=best_C, gamma=best_gamma,
    class_weight=None, probability=True, random_state=RANDOM_STATE
)
svm_balanced_prob = SVC(
    kernel="rbf", C=best_C, gamma=best_gamma,
    class_weight="balanced", probability=True, random_state=RANDOM_STATE
)

svm_default_prob.fit(X_train_s, y_train)
svm_balanced_prob.fit(X_train_s, y_train)

default_pred = svm_default_prob.predict(X_test_s)
balanced_pred = svm_balanced_prob.predict(X_test_s)

part5_rows = []
for name, pred in [("Default", default_pred), ("Balanced", balanced_pred)]:
    part5_rows.append({
        "Model": name,
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0),
        "Accuracy": accuracy_score(y_test, pred)
    })

part5_results = pd.DataFrame(part5_rows)
display(part5_results.round(4))
print("Default confusion matrix:\n", confusion_matrix(y_test, default_pred))
print("Balanced confusion matrix:\n", confusion_matrix(y_test, balanced_pred))

balanced_scores = svm_balanced_prob.predict_proba(X_test_s)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, balanced_scores)
roc_auc = roc_auc_score(y_test, balanced_scores)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Balanced SVM (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.scatter([fpr[np.argmin(np.abs(thresholds - 0.5))]],
            [tpr[np.argmin(np.abs(thresholds - 0.5))]],
            label="Threshold near 0.5")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Balanced RBF SVM")
plt.legend()
plt.show()

In [ ]:
d = part5_results.set_index("Model")
print(f"(a) Precision changed from {d.loc['Default','Precision']:.4f} to "
      f"{d.loc['Balanced','Precision']:.4f}; recall changed from "
      f"{d.loc['Default','Recall']:.4f} to {d.loc['Balanced','Recall']:.4f}.")
print("(b) Missing an actual churner (false negative) can mean losing a customer and future revenue. "
      "A false positive can waste retention-offer costs. The better deployment choice depends on the relative "
      "costs; a balanced model is attractive when identifying more churners is the priority, provided its extra "
      "false positives are affordable.")
print("(c) SVC does not provide probabilities by default. `probability=True` enables probability estimates, "
      "which are accessed with `predict_proba()` for the ROC curve.")
print(f"Balanced model ROC-AUC: {roc_auc:.4f}")

**ROC interpretation:** The ROC curve compares the true-positive rate with the false-positive rate across thresholds. A curve closer to the top-left corner and a larger AUC indicate better ranking ability. The marked point is the operating point nearest the default 0.5 probability threshold.

## Part 6 — Model Showdown: SVM vs. Everything Else

Decision Tree, tuned Random Forest, Logistic Regression, KNN, and the tuned RBF SVM are compared on the same train/test split. The Random Forest is tuned with a small cross-validation grid to keep the experiment practical.

In [ ]:
dt_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid={
        "n_estimators": [100, 200],
        "max_depth": [None, 12]
    },
    cv=3,
    scoring="f1",
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
rf_model = rf_grid.best_estimator_

lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr_model.fit(X_train_s, y_train)

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_s, y_train)

comparison_models = {
    "Decision Tree": (dt_model, X_test),
    "Random Forest (tuned)": (rf_model, X_test),
    "Logistic Regression": (lr_model, X_test_s),
    "KNN": (knn_model, X_test_s),
    "Best SVM (RBF)": (best_svm, X_test_s)
}

comparison_rows = []
for name, (model, test_data) in comparison_models.items():
    pred = model.predict(test_data)
    comparison_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1-score": f1_score(y_test, pred, zero_division=0)
    })

comparison_results = pd.DataFrame(comparison_rows).sort_values("F1-score", ascending=False).reset_index(drop=True)
display(comparison_results.round(4))
print("Best Random Forest parameters:", rf_grid.best_params_)

plt.figure(figsize=(10, 5))
sns.barplot(data=comparison_results, x="Model", y="F1-score")
plt.xticks(rotation=25, ha="right")
plt.ylim(0, 1)
plt.title("F1-score Comparison Across Five Models")
plt.tight_layout()
plt.show()

In [ ]:
f1_winner = comparison_results.loc[comparison_results["F1-score"].idxmax()]
recall_winner = comparison_results.loc[comparison_results["Recall"].idxmax()]
print(f"(a) Highest F1-score: {f1_winner['Model']} ({f1_winner['F1-score']:.4f}). "
      f"Highest recall: {recall_winner['Model']} ({recall_winner['Recall']:.4f}). "
      f"They are {'the same' if f1_winner['Model'] == recall_winner['Model'] else 'different'} models.")
print("(b) SVM can require more computation because kernel methods compare many training points. "
      "Its extra cost is worthwhile only if it provides a meaningful improvement in F1/recall over simpler models.")
print(f"(c) Based on the measured test F1-score, the leading candidate is {f1_winner['Model']}. "
      f"It has precision={f1_winner['Precision']:.4f}, recall={f1_winner['Recall']:.4f}, "
      f"and F1={f1_winner['F1-score']:.4f}. A telecom company should also consider the cost of false negatives "
      "and false positives before deployment.")

**F1-score chart interpretation:** The bar chart compares the balance between precision and recall for all five algorithms. The highest bar is the F1-score winner; the comparison table also shows whether that model has sufficient recall for churn detection.

## Part 7 — Final Predictions and Wrap-Up

The final model is selected using the highest test F1-score from Part 6. The prediction file contains the true label, predicted label, and probability-like churn score. It is saved as `svm_predictions.csv` as required by the lab manual.

In [ ]:
final_model_name = comparison_results.iloc[0]["Model"]
final_model, final_test_data = comparison_models[final_model_name]
final_predictions = final_model.predict(final_test_data)

if hasattr(final_model, "predict_proba"):
    final_scores = final_model.predict_proba(final_test_data)[:, 1]
elif hasattr(final_model, "decision_function"):
    raw_scores = final_model.decision_function(final_test_data)
    final_scores = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
else:
    final_scores = final_predictions.astype(float)

predictions_df = pd.DataFrame({
    "True_Churn": y_test.reset_index(drop=True),
    "Predicted_Churn": final_predictions,
    "Predicted_Churn_Score": final_scores
})

predictions_df.to_csv("svm_predictions.csv", index=False)
print("Final selected model:", final_model_name)
print("Saved file: svm_predictions.csv")
display(predictions_df.head(10))

In [ ]:
print("EXECUTIVE SUMMARY")
print(
    f"The Telco Customer Churn dataset was used to compare five classification algorithms. "
    f"The selected model was {final_model_name}, which achieved accuracy {final_row['Accuracy']:.4f}, "
    f"precision {final_row['Precision']:.4f}, recall {final_row['Recall']:.4f}, and "
    f"F1-score {final_row['F1-score']:.4f} on the held-out test set. "
    f"Feature scaling was important for SVM because its margin depends on feature distances. "
    f"For the RBF SVM class-weight experiment, recall changed from {default_p:.4f} without balancing "
    f"to {balanced_p:.4f} with class_weight='balanced'. "
    f"The main business trade-off is between missing real churners and incorrectly flagging loyal customers. "
    f"One limitation is that the evaluation uses a single train/test split and historical customer data, "
    f"so performance may change on future customer populations."
)
print("\nFinal Reflection:")
rbf_f1 = kernel_results.loc[kernel_results["Kernel"] == "rbf", "F1-score"].iloc[0]
linear_f1 = kernel_results.loc[kernel_results["Kernel"] == "linear", "F1-score"].iloc[0]
print(f"(a) The most surprising result in this run was that the best overall model was {final_model_name}, while the best kernel in the initial SVM comparison was {best_kernel_name}. This shows that a more complex SVM kernel is not automatically the best model for this dataset.")
print("(b) With one more hour, I would tune the probability threshold using a validation set and compare the business cost of false negatives versus false positives. I would also test feature engineering or SMOTE inside a cross-validation pipeline.")

## Final Submission Checklist

- [x] Student information block included
- [x] Part 0: data cleaning, encoding, and stratified split
- [x] Part 1: scaled vs. unscaled SVM experiment
- [x] Part 2: linear, RBF, and polynomial kernel comparison
- [x] Part 3: PCA projection and decision-boundary plots for three C values
- [x] Part 4: GridSearchCV and heatmap
- [x] Part 5: class-weight comparison and ROC curve
- [x] Part 6: five-model comparison table and F1-score chart
- [x] Part 7: predictions CSV and executive summary

**Before submission:** Confirm the notebook outputs are present after running all cells, then push both the notebook and `svm_predictions.csv` to the course GitHub repository.